In [1]:
import os
import json
import calendar
import smtplib

from datetime import datetime, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders


STORAGE_FILE = "Email_Storage.json"


def subtract_months(date_obj, months):
    """
    Subtract calendar months from a date.
    Example:
        2026-10-04 - 2 months = 2026-08-04
    """
    month_index = date_obj.month - 1 - months

    year = date_obj.year + month_index // 12
    month = month_index % 12 + 1

    day = min(
        date_obj.day,
        calendar.monthrange(year, month)[1]
    )

    return date_obj.replace(
        year=year,
        month=month,
        day=day
    )


def subtract_years(date_obj, years):
    """
    Subtract calendar years from a date.
    Handles leap years safely.
    """
    year = date_obj.year - years

    # Handle February 29
    day = min(
        date_obj.day,
        calendar.monthrange(year, date_obj.month)[1]
    )

    return date_obj.replace(
        year=year,
        day=day
    )


def get_lookback_date(look_back, look_back_format):
    """
    Returns the earliest allowed date for sending.

    Example:
        look_back=7, format='day'
        Current date = 2026-10-04
        Look-back date = 2026-09-27
    """

    today = datetime.now().date()

    if not isinstance(look_back, int) or look_back < 0:
        raise ValueError("look_back must be a non-negative integer")

    look_back_format = look_back_format.lower().strip()

    if look_back_format in ("day", "days"):
        return today - timedelta(days=look_back)

    elif look_back_format in ("month", "months"):
        return subtract_months(today, look_back)

    elif look_back_format in ("year", "years"):
        return subtract_years(today, look_back)

    else:
        raise ValueError(
            "look_back_format must be one of: day, month, year"
        )


def load_email_storage():
    """
    Load existing Email_Storage.json.

    If the file does not exist, return an empty dictionary.
    Existing data is never discarded.
    """

    if not os.path.exists(STORAGE_FILE):
        return {}

    try:
        with open(STORAGE_FILE, "r", encoding="utf-8") as file:
            data = json.load(file)

        if isinstance(data, dict):
            return data

        return {}

    except (json.JSONDecodeError, OSError):
        return {}


def save_email_storage(email_data):
    """
    Save Email_Storage.json while preserving
    all existing email records.
    """

    with open(STORAGE_FILE, "w", encoding="utf-8") as file:
        json.dump(
            email_data,
            file,
            indent=4,
            ensure_ascii=False
        )


def send_email(
    recipient_email: str,
    subject: str,
    body: str,
    sender_email: str,
    sender_password: str,
    attachments=None,
    look_back: int = 0,
    look_back_format: str = "day"
) -> bool:
    """
    Send an email with optional PDF attachments.

    Parameters:
        recipient_email (str):
            Recipient's email address.

        subject (str):
            Email subject.

        body (str):
            Email body.

        sender_email (str):
            Sender's Gmail address.

        sender_password (str):
            Gmail App Password.

        attachments (str | list[str] | None):
            Path to one PDF or a list of PDF paths.

        look_back (int):
            Number of days/months/years to look back.

            Example:
                look_back=7
                look_back_format="day"

        look_back_format (str):
            Must be one of:
                "day"
                "month"
                "year"

    Returns:
        bool:
            True  -> email was sent
            False -> email was not sent / failed
    """

    try:

        # --------------------------------------------------
        # Validate look-back parameters
        # --------------------------------------------------

        if not isinstance(look_back, int):
            raise ValueError("look_back must be an integer")

        if look_back < 0:
            raise ValueError("look_back cannot be negative")

        look_back_format = look_back_format.lower().strip()

        if look_back_format not in (
            "day",
            "days",
            "month",
            "months",
            "year",
            "years"
        ):
            raise ValueError(
                "look_back_format must be either "
                "'day', 'month', or 'year'"
            )

        # --------------------------------------------------
        # Load existing Email_Storage.json
        # --------------------------------------------------

        email_data = load_email_storage()

        today = datetime.now().date()

        # --------------------------------------------------
        # Check whether this recipient was recently emailed
        # --------------------------------------------------

        if recipient_email in email_data:

            last_sent_string = email_data[recipient_email]

            try:
                last_sent_date = datetime.strptime(
                    last_sent_string,
                    "%Y-%m-%d"
                ).date()

            except ValueError:
                # Invalid stored date -> allow sending
                last_sent_date = None

            if last_sent_date is not None:

                lookback_date = get_lookback_date(
                    look_back,
                    look_back_format
                )

                # If email was already sent within look-back period
                if last_sent_date >= lookback_date:

                    print(
                        f"Email NOT sent to {recipient_email}. "
                        f"Last sent on {last_sent_date}, "
                        f"which is within the last "
                        f"{look_back} {look_back_format}."
                    )

                    return False

        # --------------------------------------------------
        # Create email
        # --------------------------------------------------

        message = MIMEMultipart()

        message["From"] = sender_email
        message["To"] = recipient_email
        message["Subject"] = subject

        # Add email body
        message.attach(
            MIMEText(body, "plain")
        )

        # --------------------------------------------------
        # Normalize attachments
        # --------------------------------------------------

        if attachments is None:
            attachments = []

        elif isinstance(attachments, str):
            attachments = [attachments]

        # --------------------------------------------------
        # Attach PDF files
        # --------------------------------------------------

        for file_path in attachments:

            if not os.path.exists(file_path):
                raise FileNotFoundError(
                    f"Attachment not found: {file_path}"
                )

            if not file_path.lower().endswith(".pdf"):
                raise ValueError(
                    f"Only PDF files are supported: {file_path}"
                )

            with open(file_path, "rb") as file:

                pdf = MIMEBase(
                    "application",
                    "pdf"
                )

                pdf.set_payload(file.read())

            encoders.encode_base64(pdf)

            pdf.add_header(
                "Content-Disposition",
                f'attachment; filename="{os.path.basename(file_path)}"'
            )

            message.attach(pdf)

        # --------------------------------------------------
        # Connect to Gmail and send
        # --------------------------------------------------

        with smtplib.SMTP(
            "smtp.gmail.com",
            587
        ) as server:

            server.starttls()

            server.login(
                sender_email,
                sender_password
            )

            server.sendmail(
                sender_email,
                recipient_email,
                message.as_string()
            )

        # --------------------------------------------------
        # Update JSON ONLY after successful email
        # --------------------------------------------------

        email_data[recipient_email] = today.strftime(
            "%Y-%m-%d"
        )

        save_email_storage(email_data)

        print(
            f"Email sent successfully to {recipient_email}"
        )

        print(
            f"Email_Storage.json updated: {today}"
        )

        return True

    except Exception as e:

        print(
            f"Failed to send email: {e}"
        )

        return False

### Enter Your Subject Here 

In [4]:
subject = """ AI Engineer | Data Scientist | 3+ Years Experience | GenAI, RAG & Agentic AI
                         """

In [ ]:
## Enter Your Body Here 

In [5]:
body = """ Hi,
                
                I am a Data Scientist with 3 years of experience building AI and predictive models in the healthcare and automotive sectors.
                I have attached my resume for your review.
                
                My core technical skills include:
                
                Python & Statistics
                
                Machine Learning & Deep Learning
                
                Generative AI & RAG Architectures
                
                I look forward to discussing how my background
                can add value to your data science team.
                Also i have attached my resume for your reference
                
                Best regards,
                
                Rahul Singh
"""

In [7]:
## Upload You resume in same location 

In [8]:
# Make sure you enter your email-id, email-id-password

### To Get Gmail Password in Encrypt Format Visit below link

https://myaccount.google.com/apppasswords?utm_source=chatgpt.com&pli=1&rapt=AEjHL4M36AhAbTcbqBqaoegGiu4K5G9IVRDzVzfV1orCZxwz5yfZIAwPjehi0Se1KzH1xweHriu9E_JpYiRh4AiM-oHYckCrknevZSOTs7vXbqiV_z3oqEA

In [11]:
list_of_emails = ["abc@abc.com", "cdc@cdc.com"]

In [12]:
count = 0 
for email in list_of_emails:
    count+=1
    print("currently we are at",count, "----email--->")
    send_email(
    recipient_email=email,
    subject=subject,
    body=body,
    sender_email="javarahul04@gmail.com",
    sender_password="Enter Your Encrypted Password Here",
    attachments="RAHUL_SINGH_3YR_EXPERIENCE_DATA_SCIENTIST_.pdf",
    look_back=1,
    look_back_format="months"
            )

currently we are at 1 ----email--->
Failed to send email: Attachment not found: RAHUL_SINGH_3YR_EXPERIENCE_DATA_SCIENTIST_.pdf
currently we are at 2 ----email--->
Failed to send email: Attachment not found: RAHUL_SINGH_3YR_EXPERIENCE_DATA_SCIENTIST_.pdf
